# RAG-Based Gen-AI Chatbot — Q&A over YouTube Video Transcripts

This notebook builds an end-to-end **Retrieval-Augmented Generation (RAG)** chatbot that answers
questions **only** from the transcripts of selected YouTube videos. If the answer isn't in the
transcripts, the bot replies `I don't know.`

**Topic chosen:** Generative AI & Large Language Models (a set of technical talks on YouTube).

## Pipeline

```mermaid
flowchart LR
    A[YouTube Video IDs] --> B[YouTube Transcript API]
    B --> C[Save transcripts to .txt files]
    C --> D[Text Splitter - chunks + timestamps]
    D --> E[HuggingFace Embeddings]
    E --> F[FAISS Vector Store]
    F --> G[Retriever]
    G --> H[LLM - Groq/OpenAI/Gemini/HuggingFace]
    H --> I[Context-only Answer + Timestamp Citations]
    I --> J[CLI Chat Loop / Gradio UI]
```

## Notebook sections
1. Install & import libraries
2. Select topic & YouTube videos, extract video IDs
3. Fetch transcripts via YouTube Transcript API
4. Save transcripts to text files
5. Load & split transcripts into chunks (with timestamp metadata)
6. Generate embeddings & store in FAISS vector DB
7. Set up retriever for context lookup
8. Integrate LLM (dynamic: Groq / OpenAI / Gemini / free HuggingFace fallback)
9. Build RAG chain with a strict "context-only" prompt
10. Test end-to-end (in-context vs out-of-context questions)
11. Interactive CLI chat loop (type `exit` to quit)
12. Gradio chat UI

## Setup
1. `pip install -r requirements.txt`
2. Copy `.env.example` to `.env` and optionally add an API key (`GROQ_API_KEY`, `OPENAI_API_KEY`, or `GOOGLE_API_KEY`).
   No key is required — the notebook falls back to a free local HuggingFace model.
3. Run all cells top to bottom.


## 1. Install & Import Required Libraries

In [ ]:
%pip install -q youtube-transcript-api langchain langchain-community langchain-text-splitters langchain-huggingface langchain-openai langchain-groq langchain-google-genai sentence-transformers scikit-learn faiss-cpu transformers torch gradio python-dotenv

In [ ]:
import os
import re

print("Imports OK. Working directory:", os.getcwd())

## 2. Select Topic & YouTube Videos, Extract Video IDs

**Topic:** Generative AI & Large Language Models.

Add any YouTube URL (or bare 11-character video ID) to `VIDEO_URLS` below — the topic is easy to
swap (Healthcare, Data Engineering, Podcasts, etc.) by replacing this list.

In [ ]:
TOPIC = "Generative AI & Large Language Models"

# Any mix of full YouTube URLs or bare video IDs works.
VIDEO_URLS = [
    "https://www.youtube.com/watch?v=zjkBMFhNj_g",  # Intro to Large Language Models - Andrej Karpathy
    "https://www.youtube.com/watch?v=bZQun8Y4L2A",  # State of GPT - Andrej Karpathy
    "https://youtu.be/kCc8FmEb1nY",                 # Let's build GPT, from scratch - Andrej Karpathy
]

TRANSCRIPTS_DIR = "transcripts"
INDEX_DIR = "faiss_index"
os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)


def extract_video_id(url_or_id: str) -> str:
    """Extract the 11-character YouTube video ID from a URL, or pass through a bare ID."""
    url_or_id = url_or_id.strip()
    if re.fullmatch(r"[0-9A-Za-z_-]{11}", url_or_id):
        return url_or_id
    match = re.search(r"(?:v=|youtu\.be/|shorts/)([0-9A-Za-z_-]{11})", url_or_id)
    if match:
        return match.group(1)
    raise ValueError(f"Could not extract a video ID from: {url_or_id}")


VIDEO_IDS = [extract_video_id(u) for u in VIDEO_URLS]
print("Topic:", TOPIC)
print("Video IDs:", VIDEO_IDS)

## 3-4. Fetch Transcripts via YouTube Transcript API & Save to Text Files

Each transcript is saved as `transcripts/<video_id>.txt` with one caption line per row in the
format `[HH:MM:SS] caption text`, so the timestamp can later be used as a citation (the video
equivalent of a "page number").

If a video has no captions available (or there is no internet access), the notebook falls back to
a bundled sample transcript about Generative AI so the rest of the pipeline stays fully
executable.

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

try:
    from youtube_transcript_api import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
except ImportError:  # pragma: no cover - depends on installed library version
    TranscriptsDisabled = NoTranscriptFound = VideoUnavailable = Exception


def fetch_transcript(video_id: str):
    """Return a list of {'text','start','duration'} dicts.

    Supports both the classic (get_transcript) and newer (instance.fetch) API surfaces of
    youtube-transcript-api across versions.
    """
    try:
        return YouTubeTranscriptApi.get_transcript(video_id)
    except AttributeError:
        fetched = YouTubeTranscriptApi().fetch(video_id)
        return fetched.to_raw_data() if hasattr(fetched, "to_raw_data") else list(fetched)


def format_timestamp(seconds) -> str:
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


# Bundled fallback so the pipeline is fully executable even without internet/caption access.
SAMPLE_TRANSCRIPT = [
    {"start": 0, "text": "Welcome to this introduction to generative AI and large language models."},
    {"start": 9, "text": "A large language model, or LLM, is trained on huge amounts of text to predict the next token in a sequence."},
    {"start": 20, "text": "The transformer architecture, introduced in the paper Attention Is All You Need, underlies most modern LLMs."},
    {"start": 32, "text": "Transformers use a self-attention mechanism that lets the model weigh the importance of different words in context."},
    {"start": 45, "text": "Training happens in stages: pretraining on internet-scale text, followed by supervised fine-tuning."},
    {"start": 58, "text": "Reinforcement learning from human feedback, or RLHF, is often used to align model behavior with human preferences."},
    {"start": 70, "text": "Embeddings are vector representations of text that capture semantic meaning, placing similar concepts close together."},
    {"start": 83, "text": "Retrieval-Augmented Generation, or RAG, combines a retriever with a language model so answers are grounded in external documents."},
    {"start": 96, "text": "In a RAG pipeline, documents are split into chunks, embedded, and stored in a vector database such as FAISS or Chroma."},
    {"start": 110, "text": "When a user asks a question, the system retrieves the most relevant chunks and passes them to the LLM as context."},
    {"start": 123, "text": "A well designed prompt instructs the model to answer only from the provided context and avoid hallucinating facts."},
    {"start": 136, "text": "If the retrieved context does not contain the answer, a well behaved RAG chatbot should say it does not know."},
    {"start": 148, "text": "Fine-tuning adapts a pretrained model to a specific task or domain using a smaller, labeled dataset."},
    {"start": 160, "text": "Hallucination refers to a model generating plausible sounding but factually incorrect or unsupported statements."},
    {"start": 172, "text": "Prompt engineering is the practice of crafting inputs that steer a model toward the desired kind of response."},
    {"start": 184, "text": "Vector databases index embeddings so that semantically similar chunks can be retrieved quickly using similarity search."},
    {"start": 196, "text": "Popular free and open embedding models include the sentence-transformers family, such as all-MiniLM-L6-v2."},
    {"start": 208, "text": "Thanks for watching, that concludes this overview of generative AI, transformers, and retrieval augmented generation."},
]

fetched_any = False
for video_id in VIDEO_IDS:
    out_path = os.path.join(TRANSCRIPTS_DIR, f"{video_id}.txt")
    try:
        segments = fetch_transcript(video_id)
    except (TranscriptsDisabled, NoTranscriptFound, VideoUnavailable, Exception) as exc:
        print(f"Could not fetch transcript for {video_id} ({exc}). Skipping.")
        continue
    with open(out_path, "w", encoding="utf-8") as f:
        for seg in segments:
            f.write(f"[{format_timestamp(seg['start'])}] {seg['text']}\n")
    fetched_any = True
    print(f"Saved transcript -> {out_path} ({len(segments)} segments)")

if not fetched_any:
    print("No live transcripts could be fetched (no internet/captions available). Using bundled sample transcript instead.")
    VIDEO_IDS = ["sample_genai_intro"]
    out_path = os.path.join(TRANSCRIPTS_DIR, f"{VIDEO_IDS[0]}.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        for seg in SAMPLE_TRANSCRIPT:
            f.write(f"[{format_timestamp(seg['start'])}] {seg['text']}\n")
    print(f"Saved fallback transcript -> {out_path}")

## 5. Load & Split Transcripts into Chunks (with timestamp metadata)

Captions are first grouped into ~800-character timestamped blocks, then further split with
LangChain's `RecursiveCharacterTextSplitter` (500 chars, 80 overlap). Each resulting chunk keeps
the `video_id` and `timestamp` of its parent block as metadata, so answers can cite *where* in the
video the information came from.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

LINE_RE = re.compile(r"^\[(\d{2}:\d{2}:\d{2})\]\s*(.*)$")


def timestamp_to_seconds(ts: str) -> int:
    h, m, s = (int(p) for p in ts.split(":"))
    return h * 3600 + m * 60 + s


def load_transcript_lines(path: str):
    """Parse '[HH:MM:SS] text' lines back into (seconds, timestamp, text) tuples."""
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            match = LINE_RE.match(raw_line.strip())
            if match:
                ts, text = match.groups()
                lines.append((timestamp_to_seconds(ts), ts, text))
    return lines


BLOCK_CHAR_SIZE = 800  # group raw caption lines into timestamped blocks before fine-grained splitting

block_documents = []
for video_id in VIDEO_IDS:
    path = os.path.join(TRANSCRIPTS_DIR, f"{video_id}.txt")
    if not os.path.exists(path):
        continue
    buffer_text, block_start_ts, char_count = [], None, 0
    for _seconds, ts, text in load_transcript_lines(path):
        if block_start_ts is None:
            block_start_ts = ts
        buffer_text.append(text)
        char_count += len(text)
        if char_count >= BLOCK_CHAR_SIZE:
            block_documents.append(Document(
                page_content=" ".join(buffer_text),
                metadata={"video_id": video_id, "timestamp": block_start_ts, "source": path},
            ))
            buffer_text, block_start_ts, char_count = [], None, 0
    if buffer_text:
        block_documents.append(Document(
            page_content=" ".join(buffer_text),
            metadata={"video_id": video_id, "timestamp": block_start_ts, "source": path},
        ))

print(f"Built {len(block_documents)} timestamped block(s) from {len(VIDEO_IDS)} video(s).")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(block_documents)
print(f"Split into {len(chunks)} chunk(s) for embedding.")
print(chunks[0].page_content[:200])
print(chunks[0].metadata)

## 6. Generate Embeddings & Store in a Vector Database (FAISS)

Uses the free, local `sentence-transformers/all-MiniLM-L6-v2` embedding model (no API key needed)
and stores the vectors in a FAISS index persisted to disk. If that model can't be downloaded
(e.g. a network blocks huggingface.co), this falls back to a local scikit-learn TF-IDF vectorizer
so indexing still works fully offline, just with lower semantic quality. Swapping in Chroma or
Pinecone only requires changing the vector-store part of this cell.

In [ ]:
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import FAISS


class TfidfEmbeddings(Embeddings):
    """Pure scikit-learn fallback embedding model - no downloads, works fully offline."""

    def __init__(self, max_features: int = 4096):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.vectorizer = TfidfVectorizer(max_features=max_features)
        self._fitted = False

    def embed_documents(self, texts):
        matrix = self.vectorizer.fit_transform(texts)
        self._fitted = True
        return matrix.toarray().tolist()

    def embed_query(self, text):
        if not self._fitted:
            self.vectorizer.fit([text])
            self._fitted = True
        return self.vectorizer.transform([text]).toarray()[0].tolist()


def get_embedding_model():
    """Try the free HuggingFace embedding model; fall back to local TF-IDF if unreachable."""
    try:
        model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        model.embed_query("connectivity check")  # forces a real download/verify attempt
        return model
    except Exception as exc:
        print(f"Could not load HuggingFace embeddings ({exc}).")
        print("Falling back to local TF-IDF embeddings (fully offline, no downloads, lower quality).")
        return TfidfEmbeddings()


embedding_model = get_embedding_model()

vectorstore = FAISS.from_documents(chunks, embedding_model)
if not isinstance(embedding_model, TfidfEmbeddings):
    # TF-IDF vectorizer state isn't persisted, so only save/reload the index in the normal case.
    vectorstore.save_local(INDEX_DIR)
print(f"FAISS index built with {vectorstore.index.ntotal} vectors.")

## 7. Set Up Retriever for Context Lookup

The retriever performs similarity search over the FAISS index and returns the top-k most relevant
chunks (with their `video_id` / `timestamp` metadata) for a given query.

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

_sample_hits = retriever.invoke("What is a transformer model?")
for doc in _sample_hits:
    print(f"[{doc.metadata['video_id']} @ {doc.metadata['timestamp']}] {doc.page_content[:120]}...")

## 8. Integrate an LLM for Answer Generation

`get_llm()` auto-selects a provider based on which API key is present in the environment
(loaded from a local `.env` file), in this priority order:

`GROQ_API_KEY` → `OPENAI_API_KEY` → `GOOGLE_API_KEY` → free local HuggingFace model
(`google/flan-t5-base`, no key required).

This lets you plug in GPT, Grok/Groq, Gemini, or a completely free Hugging Face model without
changing any other code.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads a local .env file if present


def get_llm():
    """Pick an LLM based on whichever API key is available; falls back to a free local model."""
    if os.getenv("GROQ_API_KEY"):
        from langchain_groq import ChatGroq
        print("Using Groq (llama-3.1-8b-instant).")
        return ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI
        print("Using OpenAI (gpt-4o-mini).")
        return ChatOpenAI(model="gpt-4o-mini", temperature=0)
    if os.getenv("GOOGLE_API_KEY"):
        from langchain_google_genai import ChatGoogleGenerativeAI
        print("Using Google Gemini (gemini-3.6-flash).")
        return ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

    print("No API key found - trying the free local HuggingFace model 'google/flan-t5-base'.")
    try:
        from langchain_huggingface import HuggingFacePipeline
        from transformers import pipeline

        pipe = pipeline("text2text-generation", model="google/flan-t5-base", max_new_tokens=256)
        return HuggingFacePipeline(pipeline=pipe)
    except Exception as exc:
        print(f"Could not load local HuggingFace model ({exc}).")
        print("Falling back to extractive mode: answers will be the most relevant transcript "
              "excerpt instead of an LLM-generated response.")
        return None


llm = get_llm()

## 9. Build the RAG Chain with a Context-Only Constraint

The prompt strictly instructs the model to answer only from the retrieved transcript chunks and to
reply `I don't know.` when the context doesn't contain the answer. `ask_with_sources()` also
returns the `video_id @ timestamp` of each chunk used, as a citation (the video equivalent of a
page number). If no LLM is configured, or the LLM call fails at runtime (rate limit, network
error, etc.), it gracefully falls back to `ask_extractive()`, which returns the closest transcript
excerpt instead of hanging or crashing.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a Q&A assistant that answers questions ONLY using the transcript context below.

Rules:
- Use ONLY the information contained in the context to answer the question.
- If the answer is not present in the context, reply with exactly: I don't know.
- Do not use any outside knowledge or make assumptions beyond the context.
- Keep the answer concise and factual.

Context:
{context}

Question: {question}

Answer:"""
)


def format_docs(docs) -> str:
    return "\n\n".join(
        f"[Source: {d.metadata['video_id']} @ {d.metadata['timestamp']}]\n{d.page_content}"
        for d in docs
    )


_answer_chain = RAG_PROMPT | llm | StrOutputParser() if llm is not None else None


def ask_extractive(question: str, distance_threshold: float = 1.3):
    """No-LLM fallback: return the closest transcript excerpt instead of a generated answer."""
    hits = vectorstore.similarity_search_with_score(question, k=4)
    if not hits or hits[0][1] > distance_threshold:
        return "I don't know.", []
    doc, _score = hits[0]
    excerpt = doc.page_content.strip()[:400]
    answer = f"(Extractive mode - no LLM available) Closest transcript excerpt: \"{excerpt}\""
    sources = [f"{d.metadata['video_id']} @ {d.metadata['timestamp']}" for d, _ in hits]
    return answer, sources


def ask_with_sources(question: str):
    """Retrieve relevant chunks, generate a context-only answer, and return (answer, sources)."""
    if _answer_chain is None:
        return ask_extractive(question)
    docs = retriever.invoke(question)
    context = format_docs(docs)
    try:
        answer = _answer_chain.invoke({"context": context, "question": question}).strip()
    except Exception as exc:
        print(f"LLM call failed ({exc}); falling back to extractive mode for this question.")
        answer, sources = ask_extractive(question)
        return f"\u26A0\uFE0F (LLM unavailable, showing closest excerpt instead) {answer}", sources
    sources = [] if answer == "I don't know." else [
        f"{d.metadata['video_id']} @ {d.metadata['timestamp']}" for d in docs
    ]
    return answer, sources

## 10. Test the Chatbot End-to-End

One question that should be answerable from the transcripts, and one that clearly is not — to
verify the "context-only" / `I don't know.` behavior.

In [ ]:
test_questions = [
    "What is a large language model?",   # in-context -> should be answered with a citation
    "What is the capital of France?",    # out-of-context -> should reply "I don't know."
]

for q in test_questions:
    answer, sources = ask_with_sources(q)
    print(f"Q: {q}\nA: {answer}")
    if sources:
        print(f"Sources: {sources}")
    print("-" * 60)

## 11. Interactive Chat Loop (type `exit` to quit)

Run the cell below, then call `run_cli_chat()` to chat in a loop directly in this notebook. The
loop keeps asking for input until you type `exit`.

In [ ]:
def run_cli_chat():
    print(f"YouTube RAG Chatbot ready. Topic: {TOPIC}")
    print("Type your question, or 'exit' to quit.\n")
    while True:
        question = input("You: ").strip()
        if question.lower() == "exit":
            print("Bot: Goodbye!")
            break
        if not question:
            continue
        answer, sources = ask_with_sources(question)
        print(f"Bot: {answer}")
        if sources:
            print(f"     Sources: {', '.join(sources)}")


# Uncomment to chat right here in the notebook (type 'exit' to stop):
# run_cli_chat()

## 12. Gradio Chat UI

A richer `gr.Blocks` chat UI (matching `app.py`) wired to the same `ask_with_sources()` function,
launched inline in the notebook:
- Shows a welcome message with a short AI-generated summary of the loaded video(s) before any
  question is asked, so you know what topics you can ask about.
- Small chat font, simple colored avatars, and no raw "Sources:" text cluttering the answer.
- Typing `exit` ends the session with a goodbye message; **New Chat** starts a fresh one.

Close/stop the cell to shut the UI down.

In [ ]:
import base64

import gradio as gr

CHAT_CSS = "#nb_chatbot, #nb_chatbot * { font-size: 13px !important; }"


def _nb_avatar(emoji: str, bg: str) -> str:
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 64 64">'
        f'<circle cx="32" cy="32" r="32" fill="{bg}"/>'
        f'<text x="32" y="44" font-size="34" text-anchor="middle">{emoji}</text></svg>'
    )
    return "data:image/svg+xml;base64," + base64.b64encode(svg.encode("utf-8")).decode("ascii")


NB_USER_AVATAR = _nb_avatar("\U0001F9D1", "#6366f1")  # 🧑 on indigo
NB_BOT_AVATAR = _nb_avatar("\U0001F3AC", "#ec4899")  # 🎬 on pink


def _as_text(content) -> str:
    """Gradio's Chatbot may normalize string content into a list of content blocks."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(b.get("text", "") for b in content if isinstance(b, dict))
    return str(content)


def _welcome_summary() -> str:
    """Short blurb on what the loaded video(s) cover, shown before any question is asked."""
    preview_text = " ".join(c.page_content for c in chunks[:5])[:2000]
    if llm is not None:
        try:
            prompt = (
                "In 2-3 short sentences, summarize what topics this video transcript covers "
                "so a user knows what they could ask questions about. Don't answer any "
                "question, just summarize the topic.\n\nTranscript excerpt:\n" + preview_text
            )
            result = llm.invoke(prompt)
            content = result.content if hasattr(result, "content") else result
            if isinstance(content, list):
                content = "".join(b if isinstance(b, str) else b.get("text", "") for b in content)
            return str(content).strip()
        except Exception as exc:
            print(f"Could not summarize video ({exc}); falling back to a plain excerpt.")
    return preview_text[:280].strip() + "..."


def user_submit(message, history):
    history = history or []
    history.append({"role": "user", "content": message})
    return "", history


def bot_respond(history):
    question = _as_text(history[-1]["content"])
    if question.strip().lower() == "exit":
        history.append({
            "role": "assistant",
            "content": "\U0001F44B Goodbye! This chat session has ended. Click **New Chat** below to start again.",
        })
        return history, gr.update(interactive=False, placeholder="Chat ended - click 'New Chat' to continue"), gr.update(interactive=False)

    try:
        answer, _sources = ask_with_sources(question)
    except Exception as exc:
        answer = f"\u26A0\uFE0F Something went wrong answering that question: {exc}"
    history.append({"role": "assistant", "content": answer})
    return history, gr.update(), gr.update()


def new_chat():
    welcome = (
        f"\U0001F44B Welcome! I've loaded video(s): {', '.join(VIDEO_IDS)}. {_welcome_summary()} "
        "Feel free to ask me anything about it!"
    )
    return (
        [{"role": "assistant", "content": welcome}],
        gr.update(interactive=True, placeholder="Ask something about the loaded video(s)...", value=""),
        gr.update(interactive=True),
    )


with gr.Blocks(title="YouTube RAG Chatbot") as demo:
    gr.Markdown(f"### \U0001F3AC YouTube RAG Chatbot — {TOPIC}")
    chatbot = gr.Chatbot(
        label=None, show_label=False, height=420, elem_id="nb_chatbot",
        avatar_images=(NB_USER_AVATAR, NB_BOT_AVATAR),
    )
    msg = gr.Textbox(label=None, show_label=False, placeholder="Ask something about the loaded video(s)...")
    with gr.Row():
        send_btn = gr.Button("\u27A4 Send", variant="primary")
        new_chat_btn = gr.Button("\U0001F195 New Chat")
    gr.Examples(
        examples=["What is a large language model?", "What is the capital of France?"],
        inputs=msg,
        label="Try an example",
    )

    demo.load(new_chat, None, [chatbot, msg, send_btn])
    msg.submit(user_submit, [msg, chatbot], [msg, chatbot]).then(bot_respond, chatbot, [chatbot, msg, send_btn])
    send_btn.click(user_submit, [msg, chatbot], [msg, chatbot]).then(bot_respond, chatbot, [chatbot, msg, send_btn])
    new_chat_btn.click(new_chat, None, [chatbot, msg, send_btn])

demo.launch(inline=True, share=False, css=CHAT_CSS)

## Notes & Possible Extensions

- **Swap topic**: change `VIDEO_URLS` in section 2 to any videos (Healthcare, Data Engineering,
  podcasts, etc.).
- **Swap vector DB**: replace the FAISS cell in section 6 with `Chroma.from_documents(...)` or a
  Pinecone client — the rest of the pipeline (retriever, chain, UI) is unchanged.
- **Swap LLM**: set `GROQ_API_KEY`, `OPENAI_API_KEY`, or `GOOGLE_API_KEY` in `.env` to use that
  provider instead of the free local HuggingFace fallback.
- **Limitations**: some videos disable captions entirely; the free local LLM fallback
  (`flan-t5-base`) is much weaker than GPT/Groq/Gemini, so answers may be terser.